# Cu FEFF encoder study

This exploratory workflow compares compact, supervised Cu FEFF encoders with local features from the pinned M3GNet model.

| family | feature | members |
|---|---|---:|
| learned | 96D and 128D node encoders with attention readout | 2 encoder seeds x 3 heads |
| learned local | absorbing site + inverse-distance neighbors within 5 Angstrom | 4 feature sets x 3 heads |
| learned local | absorbing site + separate 0-3 and 3-5 Angstrom shells | 4 feature sets x 3 heads |
| frozen M3GNet | the same two local pooling rules over concatenated block 1-3 states | 3 heads each |

The notebook uses the published material-level train/validation/test splits. Encoder checkpoints, head checkpoints, and ensemble choices are selected only by validation error; test eta is reported afterward. Generated features record their row IDs and source checkpoint so stale artifacts fail clearly instead of being mixed.

This is a long GPU experiment and is not directly comparable to the paper's standard 64D ExpertXAS pipeline.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import dgl
import lightning.pytorch as pl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from matgl import load_model
from matgl.config import DEFAULT_ELEMENTS
from matgl.ext.pymatgen import Structure2Graph
from matgl.graph.compute import compute_pair_vector_and_distance, compute_theta_and_phi, create_line_graph
from matgl.layers import (
    MLP as M3GNetMLP,
    ActivationFunction,
    BondExpansion,
    EmbeddingBlock,
    GatedMLP,
    M3GNetBlock,
    SphericalBesselWithHarmonics,
    ThreeBodyInteractions,
)
from matgl.utils.cutoff import polynomial_cutoff
from pymatgen.core import Structure
from torch import nn
from torch.utils.data import DataLoader, Dataset

import matgl.layers._basis as matgl_basis
import matgl.layers._three_body as matgl_three_body
import matgl.utils.maths as matgl_math

from omnixas.data.ml_data import MLData, MLSplits
from omnixas.featurizer.m3gnet_featurizer import M3GNetFeaturizer
from omnixas.model.xasblock import XASBlock
from omnixas.model.xasblock_regressor import XASBlockRegressor

REPO_ROOT = Path.cwd().resolve()
while not ((REPO_ROOT / "pyproject.toml").exists() and (REPO_ROOT / "omnixas").is_dir()):
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError("Run this notebook from inside the OmniXAS repository.")
    REPO_ROOT = REPO_ROOT.parent

TASK = "Cu_FEFF"
DATA_DIR = REPO_ROOT / "tutorial_omnixas" / "ml_data"
ID_DIR = REPO_ROOT / "tutorial_omnixas" / "material_id_and_site"
RAW_ROOT = Path(os.environ.get("OMNIXAS_DATA_ROOT", REPO_ROOT.parent / "OmniXAS_data")) / "materialscloud_omnixas_raw" / "extracted"
M3GNET_DIR = REPO_ROOT / "models" / "M3GNet-MP-2021.2.8-PES"
OUT_ROOT = REPO_ROOT / "output" / "training" / "cuFeffEncoderStudy"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

if not RAW_ROOT.is_dir():
    raise FileNotFoundError(f"Missing raw OmniXAS data: {RAW_ROOT}")
if not M3GNET_DIR.is_dir():
    raise FileNotFoundError(f"Missing pinned M3GNet model: {M3GNET_DIR}")

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("medium")

SPLITS = ("train", "val", "test")
split_ids = {
    split: [line for line in (ID_DIR / f"{TASK}_{split}.txt").read_text().splitlines() if line]
    for split in SPLITS
}
y_true = {
    split: np.atleast_2d(np.loadtxt(DATA_DIR / f"{TASK}_{split}_y.txt", dtype=np.float32))
    for split in SPLITS
}

for split in SPLITS:
    if len(set(split_ids[split])) != len(split_ids[split]):
        raise ValueError(f"{split}: duplicate material/site IDs")
    if y_true[split].shape != (len(split_ids[split]), 141):
        raise ValueError(f"{split}: expected {len(split_ids[split])} x 141 targets, got {y_true[split].shape}")
    if not np.isfinite(y_true[split]).all():
        raise ValueError(f"{split}: targets contain non-finite values")

materials = {split: {row.rsplit("_", 1)[0] for row in split_ids[split]} for split in SPLITS}
for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
    overlap = materials[left] & materials[right]
    if overlap:
        raise ValueError(f"Material leakage between {left} and {right}: {sorted(overlap)[:5]}")

missing_poscars = sorted(
    material_id for material_id in set().union(*materials.values())
    if not (RAW_ROOT / "FEFF" / "Cu" / material_id / "POSCAR").is_file()
)
if missing_poscars:
    raise FileNotFoundError(f"Missing Cu FEFF POSCAR files for {missing_poscars[:5]}")

FEATURE_SCALE = 1000.0
BATCH_SIZE = 32
NUM_WORKERS = 4
CUTOFF = 5.0
N_BLOCKS = 3
GNN_DROPOUT = 0.1
ENCODER_WIDTHS = (96, 128)
ENCODER_SEEDS = (42, 43)
ENCODER_EPOCHS = 150
ENCODER_LR = 1e-3

HEAD_DIMS = [600, 600, 400]
HEAD_EPOCHS = 400
HEAD_CONFIGS = (
    {"seed": 42, "dropout": 0.50, "lr": 1e-3},
    {"seed": 44, "dropout": 0.30, "lr": 1e-3},
    {"seed": 45, "dropout": 0.25, "lr": 7e-4},
)

LEARNED_POOLINGS = ("attention", "site_weighted5", "site_shells_3_5_weighted")
FROZEN_FEATURES = {
    "frozen_site_weighted5": 384,
    "frozen_site_shells_3_5_weighted": 576,
}
PAPER_EXPERT_ETA = 5.19
OLD_V1_VAL_ETA = 7.48
OLD_V1_TEST_ETA = 7.64

TRAIN_LEARNED_ENCODERS = True
BUILD_FROZEN_FEATURES = True
TRAIN_HEADS = True

print("repo:", REPO_ROOT)
print("device:", DEVICE)
print("rows:", {split: len(rows) for split, rows in split_ids.items()})


In [ ]:
TRAIN_MEAN = y_true["train"].mean(axis=0, keepdims=True)


def eta(pred, target):
    pred = np.asarray(pred)
    if pred.shape != target.shape:
        raise ValueError(f"Prediction shape {pred.shape} does not match target shape {target.shape}")
    if not np.isfinite(pred).all():
        raise ValueError("Predictions contain non-finite values")
    model_median_mse = float(np.median(np.mean((target - pred) ** 2, axis=1)))
    if model_median_mse <= 0:
        raise ValueError(f"Expected positive median MSE, got {model_median_mse}")
    baseline_median_mse = float(np.median(np.mean((target - TRAIN_MEAN) ** 2, axis=1)))
    return baseline_median_mse / model_median_mse


# MatGL 0.8.5 creates a few basis/scatter tensors on CPU. Keep this experiment-local
# compatibility patch until the pinned MatGL stack is upgraded.
def patch_matgl_gpu_constants():
    def spherical_bessel(self, r):
        cutoff = torch.as_tensor(self.cutoff, dtype=r.dtype, device=r.device)
        roots = matgl_basis.SPHERICAL_BESSEL_ROOTS[: self.max_l, : self.max_n].to(r.device, dtype=r.dtype)
        factor = torch.sqrt(torch.as_tensor(2.0, dtype=r.dtype, device=r.device) / cutoff**3)
        r = r.clamp(max=cutoff)
        return torch.cat([
            self.funcs[i](r[:, None] * roots[i][None, :] / cutoff)
            * factor
            / torch.abs(self.funcs[i + 1](roots[i][None, :]))
            for i in range(self.max_l)
        ], dim=1)

    def combine_basis(sbf, shf, max_n, max_l, use_phi):
        if sbf.size(0) == 0:
            return sbf
        if use_phi:
            repeats = torch.repeat_interleave(2 * torch.arange(max_l, device=sbf.device) + 1, max_n)
            block_sizes = 2 * torch.arange(max_l, device=sbf.device) + 1
        else:
            repeats = torch.ones(max_l * max_n, dtype=torch.long, device=sbf.device)
            block_sizes = [1] * max_l
        columns = torch.arange(shf.size(1), device=shf.device)
        indices, start = [], 0
        for block_size in block_sizes:
            block_size = int(block_size)
            indices.append(torch.tile(columns[start : start + block_size], [max_n]))
            start += block_size
        sbf = torch.repeat_interleave(sbf, repeats, dim=1)
        shf = torch.index_select(shf, 1, torch.cat(indices))
        width = max_n * max_l * (max_l if use_phi else 1)
        return (sbf * shf).reshape(-1, width)

    def scatter_sum(x, segment_ids, num_segments, dim):
        segment_ids = matgl_math.broadcast(segment_ids.to(x.device), x, dim)
        size = list(x.size())
        size[dim] = num_segments if segment_ids.numel() else 0
        return torch.zeros(size, dtype=x.dtype, device=x.device).scatter_add_(dim, segment_ids, x)

    matgl_basis.SphericalBesselFunction._call_sbf = spherical_bessel
    matgl_basis.combine_sbf_shf = combine_basis
    matgl_three_body.combine_sbf_shf = combine_basis
    matgl_math.scatter_sum = scatter_sum
    matgl_three_body.scatter_sum = scatter_sum


class FEFFDataset(Dataset):
    def __init__(self, split):
        self.rows = []
        for row_id, spectrum in zip(split_ids[split], y_true[split], strict=True):
            material_id, site = row_id.rsplit("_", 1)
            self.rows.append((material_id, int(site), torch.from_numpy(spectrum)))
        self.structures = {}

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        material_id, site, spectrum = self.rows[index]
        if material_id not in self.structures:
            path = RAW_ROOT / "FEFF" / "Cu" / material_id / "POSCAR"
            self.structures[material_id] = Structure.from_file(path)
        structure = self.structures[material_id]
        if not 0 <= site < len(structure):
            raise IndexError(f"Absorbing site {site} is outside {material_id} with {len(structure)} sites")
        return structure, site, spectrum


class CollateGraphs:
    def __init__(self, encoder):
        self.converter = Structure2Graph(encoder.element_types, encoder.cutoff)

    def __call__(self, batch):
        graphs, sites, spectra = [], [], []
        node_offset = 0
        for structure, site, spectrum in batch:
            converted = self.converter.get_graph(structure)
            graph = converted[0]
            lattice = structure.lattice.matrix if len(converted) == 2 else converted[1]
            lattice = lattice[0] if getattr(lattice, "ndim", 0) == 3 else lattice
            lattice = torch.tensor(np.array(lattice, copy=True), dtype=torch.float32)

            if "pbc_offshift" in graph.edata:
                graph.edata["pbc_offshift"] = graph.edata["pbc_offshift"].float()
            else:
                graph.edata["pbc_offshift"] = graph.edata["pbc_offset"].float() @ lattice

            if "pos" in graph.ndata:
                graph.ndata["pos"] = graph.ndata["pos"].float()
            elif "frac_coords" in graph.ndata:
                graph.ndata["pos"] = graph.ndata["frac_coords"].float() @ lattice
            else:
                graph.ndata["pos"] = torch.as_tensor(structure.cart_coords, dtype=torch.float32)

            graph.edata["bond_vec"], graph.edata["bond_dist"] = compute_pair_vector_and_distance(graph)
            graphs.append(graph)
            sites.append(node_offset + site)
            spectra.append(spectrum)
            node_offset += graph.num_nodes()

        return {
            "graph": dgl.batch(graphs),
            "site": torch.tensor(sites, dtype=torch.long),
            "y": torch.stack(spectra).float(),
        }


patch_matgl_gpu_constants()


In [ ]:
class AttentionReadout(nn.Module):
    def __init__(self, dim, cutoff):
        super().__init__()
        self.dim, self.cutoff = dim, cutoff
        self.query = nn.Linear(dim, dim, bias=False)
        self.key = nn.Linear(dim, dim, bias=False)
        self.value = nn.Linear(dim, dim, bias=False)

    def forward(self, graph, node, sites):
        source, destination, edge_id = graph.in_edges(sites, form="all")
        batch_row = torch.zeros(graph.num_nodes(), dtype=torch.long, device=node.device)
        batch_row[sites] = torch.arange(len(sites), device=node.device)
        batch_row = batch_row[destination]
        distance = graph.edata["bond_dist"][edge_id]

        logits = (self.query(node[destination]) * self.key(node[source])).sum(-1) / self.dim**0.5
        logits += torch.log(polynomial_cutoff(distance, self.cutoff).clamp_min(1e-9))
        peak = node.new_full((len(sites),), -torch.inf).scatter_reduce(0, batch_row, logits, "amax", include_self=False)
        weight = torch.exp(logits - peak[batch_row])
        norm = node.new_zeros(len(sites)).scatter_add(0, batch_row, weight).clamp_min(1e-12)
        context = node.new_zeros(len(sites), self.dim).index_add(
            0, batch_row, (weight / norm[batch_row]).unsqueeze(-1) * self.value(node[source])
        )

        coordination = node.new_zeros(len(sites)).scatter_add(0, batch_row, torch.ones_like(distance))
        coordination3 = node.new_zeros(len(sites)).scatter_add(0, batch_row, (distance <= 3.0).float())
        min_distance = node.new_full((len(sites),), self.cutoff).scatter_reduce(0, batch_row, distance, "amin")
        mean_distance = node.new_zeros(len(sites)).scatter_add(0, batch_row, distance) / coordination.clamp_min(1)
        geometry = torch.stack([coordination / 10, coordination3 / 10, min_distance / self.cutoff, mean_distance / self.cutoff], dim=-1)
        return torch.cat([node[sites], context, geometry], dim=-1)


class XASEncoder(nn.Module):
    def __init__(self, dim):
        super().__init__()
        activation = ActivationFunction["swish"].value()
        degree = 9
        self.dim = dim
        self.element_types = DEFAULT_ELEMENTS
        self.cutoff = self.threebody_cutoff = CUTOFF
        self.bond_expansion = BondExpansion(3, 3, CUTOFF)
        self.basis_expansion = SphericalBesselWithHarmonics(3, 3, CUTOFF, use_smooth=False, use_phi=False)
        self.embedding = EmbeddingBlock(
            degree_rbf=degree, dim_node_embedding=dim, dim_edge_embedding=dim,
            ntypes_node=len(DEFAULT_ELEMENTS), activation=activation,
        )
        self.three_body_interactions = nn.ModuleList([
            ThreeBodyInteractions(
                update_network_atom=M3GNetMLP(dims=[dim, degree], activation=nn.Sigmoid(), activate_last=True),
                update_network_bond=GatedMLP(in_feats=degree, dims=[dim], use_bias=False),
            ) for _ in range(N_BLOCKS)
        ])
        self.graph_layers = nn.ModuleList([
            M3GNetBlock(
                degree=degree, activation=activation, conv_hiddens=[dim, dim],
                dim_node_feats=dim, dim_edge_feats=dim, dropout=GNN_DROPOUT,
            ) for _ in range(N_BLOCKS)
        ])
        self.readout = AttentionReadout(dim, CUTOFF)
        self.out_dim = 2 * dim + 4

    def node_features(self, graph):
        graph.edata["rbf"] = self.bond_expansion(graph.edata["bond_dist"])
        line_graph = create_line_graph(graph.to("cpu"), self.threebody_cutoff).to(graph.device)
        line_graph.apply_edges(compute_theta_and_phi)
        basis = self.basis_expansion(line_graph)
        cutoff = polynomial_cutoff(graph.edata["bond_dist"], self.threebody_cutoff)
        node, edge, state = self.embedding(graph.ndata["node_type"], graph.edata["rbf"], None)
        for three_body, graph_layer in zip(self.three_body_interactions, self.graph_layers, strict=True):
            edge = three_body(graph, line_graph, basis, cutoff, node, edge)
            edge, node, state = graph_layer(graph, edge, node, state)
        return node

    def pooled_features(self, graph, sites):
        node = self.node_features(graph)
        source, destination, edge_id = graph.in_edges(sites, form="all")
        batch_row = torch.zeros(graph.num_nodes(), dtype=torch.long, device=node.device)
        batch_row[sites] = torch.arange(len(sites), device=node.device)
        batch_row = batch_row[destination]
        distance = graph.edata["bond_dist"][edge_id]

        def weighted_neighbors(mask):
            rows = batch_row[mask]
            weights = 1.0 / distance[mask].clamp_min(1e-6)
            norm = node.new_zeros(len(sites)).scatter_add(0, rows, weights).clamp_min(1e-12)
            return node.new_zeros(len(sites), self.dim).index_add(
                0, rows, (weights / norm[rows]).unsqueeze(-1) * node[source[mask]]
            )

        return {
            "attention": self.readout(graph, node, sites),
            "site_weighted5": torch.cat([node[sites], weighted_neighbors(distance <= 5.0)], dim=-1),
            "site_shells_3_5_weighted": torch.cat([
                node[sites], weighted_neighbors(distance <= 3.0),
                weighted_neighbors((distance > 3.0) & (distance <= 5.0)),
            ], dim=-1),
        }

    def forward(self, graph, sites):
        return self.pooled_features(graph, sites)["attention"]


class LitEncoder(pl.LightningModule):
    def __init__(self, dim):
        super().__init__()
        self.encoder = XASEncoder(dim)
        self.head = nn.Sequential(
            nn.Linear(self.encoder.out_dim, 128), nn.BatchNorm1d(128), nn.SiLU(), nn.Dropout(0.25),
            nn.Linear(128, 128), nn.BatchNorm1d(128), nn.SiLU(), nn.Dropout(0.25),
            nn.Linear(128, 141), nn.Softplus(),
        )
        self.val_mses = []

    def step(self, batch, split):
        graph = batch["graph"].to(self.device)
        sites = batch["site"].to(self.device)
        target = batch["y"].to(self.device)
        pred = self.head(self.encoder(graph, sites) * FEATURE_SCALE)
        loss = ((pred - target) ** 2).mean()
        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite {split} loss")
        self.log(f"{split}_loss", loss, on_epoch=True, prog_bar=True)
        if split == "val":
            self.val_mses.append(((pred - target) ** 2).mean(dim=1).detach())
        return loss

    def training_step(self, batch, _):
        return self.step(batch, "train")

    def on_validation_epoch_start(self):
        self.val_mses.clear()

    def validation_step(self, batch, _):
        return self.step(batch, "val")

    def on_validation_epoch_end(self):
        if self.val_mses:
            self.log("val_median_mse", torch.cat(self.val_mses).median(), prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=ENCODER_LR, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, ENCODER_EPOCHS, eta_min=1e-6)
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "interval": "epoch"}}


In [ ]:
train_dataset = FEFFDataset("train")
val_dataset = FEFFDataset("val")

for width in ENCODER_WIDTHS:
    for seed in ENCODER_SEEDS:
        run_dir = OUT_ROOT / f"learned_{width}d_seed{seed}"
        checkpoint_dir = run_dir / "encoder_checkpoints"
        config_path = run_dir / "config.json"
        run_config = {
            "task": TASK,
            "encoder_width": width,
            "seed": seed,
            "cutoff": CUTOFF,
            "n_blocks": N_BLOCKS,
            "gnn_dropout": GNN_DROPOUT,
            "epochs": ENCODER_EPOCHS,
            "learning_rate": ENCODER_LR,
            "feature_scale": FEATURE_SCALE,
        }

        if run_dir.exists() and not config_path.exists():
            raise RuntimeError(f"Unversioned encoder run exists; remove or migrate it: {run_dir}")
        if config_path.exists() and json.loads(config_path.read_text()) != run_config:
            raise RuntimeError(f"Encoder configuration changed; use a new run directory: {run_dir}")

        if (checkpoint_dir / "DONE").exists():
            print("encoder cached:", run_dir.name)
            continue
        if not TRAIN_LEARNED_ENCODERS:
            raise FileNotFoundError(f"Missing completed encoder run: {run_dir}")

        run_dir.mkdir(parents=True, exist_ok=True)
        config_path.write_text(json.dumps(run_config, indent=2) + "\n")
        pl.seed_everything(seed, workers=True)
        model = LitEncoder(width)
        collate = CollateGraphs(model.encoder)
        checkpoint = ModelCheckpoint(
            checkpoint_dir,
            filename="best-{epoch:03d}-{val_median_mse:.5f}",
            monitor="val_median_mse",
            mode="min",
            save_top_k=1,
            save_last=True,
        )
        trainer = pl.Trainer(
            max_epochs=ENCODER_EPOCHS,
            accelerator="auto",
            devices=1,
            callbacks=[checkpoint],
            logger=CSVLogger(str(run_dir), name="encoder_logs", version=0),
            log_every_n_steps=10,
        )
        last_checkpoint = checkpoint_dir / "last.ckpt"
        trainer.fit(
            model,
            DataLoader(train_dataset, BATCH_SIZE, shuffle=True, collate_fn=collate, num_workers=NUM_WORKERS),
            DataLoader(val_dataset, BATCH_SIZE, shuffle=False, collate_fn=collate, num_workers=NUM_WORKERS),
            ckpt_path=str(last_checkpoint) if last_checkpoint.exists() else None,
        )
        best_checkpoint = Path(checkpoint.best_model_path)
        best_score = checkpoint.best_model_score
        if not best_checkpoint.is_file() or best_score is None or not torch.isfinite(best_score):
            raise RuntimeError(f"Encoder did not produce a finite validation-best checkpoint: {run_dir}")
        (checkpoint_dir / "DONE").write_text("ok\n")
        torch.cuda.empty_cache()

for width in ENCODER_WIDTHS:
    for seed in ENCODER_SEEDS:
        run_dir = OUT_ROOT / f"learned_{width}d_seed{seed}"
        checkpoint_dir = run_dir / "encoder_checkpoints"
        if not (checkpoint_dir / "DONE").exists():
            raise FileNotFoundError(f"Incomplete encoder run: {run_dir}")
        best_checkpoints = list(checkpoint_dir.glob("best-*.ckpt"))
        if len(best_checkpoints) != 1:
            raise RuntimeError(f"Expected one validation-best checkpoint in {checkpoint_dir}, found {len(best_checkpoints)}")
        checkpoint = best_checkpoints[0]
        checkpoint_stamp = np.int64(checkpoint.stat().st_mtime_ns)
        feature_file = run_dir / "features.npz"
        pooling_dims = {"attention": 2 * width + 4, "site_weighted5": 2 * width, "site_shells_3_5_weighted": 3 * width}

        if feature_file.exists():
            with np.load(feature_file, allow_pickle=False) as cached:
                cache_ok = int(cached["checkpoint_stamp"]) == int(checkpoint_stamp)
                cache_ok &= all(
                    f"{pooling}_{split}" in cached
                    and cached[f"{pooling}_{split}"].shape == (len(split_ids[split]), pooling_dims[pooling])
                    and np.isfinite(cached[f"{pooling}_{split}"]).all()
                    for pooling in LEARNED_POOLINGS for split in SPLITS
                )
                cache_ok &= all(
                    f"ids_{split}" in cached and np.array_equal(cached[f"ids_{split}"], split_ids[split])
                    for split in SPLITS
                )
            if not cache_ok:
                raise RuntimeError(f"Stale or invalid learned feature cache: {feature_file}")
            print("features cached:", run_dir.name)
            continue

        state = torch.load(checkpoint, map_location="cpu")["state_dict"]
        model = LitEncoder(width)
        model.load_state_dict(state)
        encoder = model.encoder.to(DEVICE).eval()
        collate = CollateGraphs(encoder)
        arrays = {f"ids_{split}": np.asarray(split_ids[split]) for split in SPLITS}

        with torch.no_grad():
            for split in SPLITS:
                chunks = {pooling: [] for pooling in LEARNED_POOLINGS}
                loader = DataLoader(
                    FEFFDataset(split),
                    BATCH_SIZE,
                    shuffle=False,
                    collate_fn=collate,
                    num_workers=NUM_WORKERS,
                )
                for batch in loader:
                    pooled = encoder.pooled_features(batch["graph"].to(DEVICE), batch["site"].to(DEVICE))
                    for pooling in LEARNED_POOLINGS:
                        chunks[pooling].append((pooled[pooling] * FEATURE_SCALE).cpu().numpy())
                for pooling, values in chunks.items():
                    features = np.concatenate(values)
                    if features.shape != (len(split_ids[split]), pooling_dims[pooling]) or not np.isfinite(features).all():
                        raise RuntimeError(f"Invalid exported features: {run_dir.name} {pooling} {split}")
                    arrays[f"{pooling}_{split}"] = features

        np.savez_compressed(feature_file, checkpoint_stamp=checkpoint_stamp, **arrays)
        del model, encoder
        torch.cuda.empty_cache()
        print("wrote", run_dir.name, {key: value.shape for key, value in arrays.items() if not key.startswith("ids_")})


In [ ]:
frozen_dir = OUT_ROOT / "frozen_features"
frozen_dir.mkdir(exist_ok=True)

if BUILD_FROZEN_FEATURES:
    m3gnet = load_model(str(M3GNET_DIR)).model.eval()
    featurizers = {depth: M3GNetFeaturizer(model=m3gnet, n_blocks=depth) for depth in (1, 2, 3)}
    zero_neighbors = np.zeros(192, dtype=np.float32)

    for split in SPLITS:
        output_paths = {name: frozen_dir / f"{name}_{split}.npz" for name in FROZEN_FEATURES}
        if all(path.exists() for path in output_paths.values()):
            for name, output_path in output_paths.items():
                with np.load(output_path, allow_pickle=False) as cached:
                    valid = cached["X"].shape == (len(split_ids[split]), FROZEN_FEATURES[name])
                    valid &= np.isfinite(cached["X"]).all()
                    valid &= np.array_equal(cached["ids"], split_ids[split])
                    valid &= str(cached["source"]) == str(M3GNET_DIR.resolve())
                if not valid:
                    raise RuntimeError(f"Stale or invalid frozen feature cache: {output_path}")
            print("frozen features cached:", split)
            continue
        if any(path.exists() for path in output_paths.values()):
            raise RuntimeError(f"Incomplete frozen feature cache for {split}; remove {frozen_dir} and rebuild")

        rows = {name: [] for name in FROZEN_FEATURES}
        material_cache = {}
        for row_number, row_id in enumerate(split_ids[split], 1):
            material_id, site_text = row_id.rsplit("_", 1)
            site = int(site_text)
            if material_id not in material_cache:
                structure = Structure.from_file(RAW_ROOT / "FEFF" / "Cu" / material_id / "POSCAR")
                states = [featurizers[depth].featurize(structure) * FEATURE_SCALE for depth in (1, 2, 3)]
                node_features = np.concatenate(states, axis=1)
                if node_features.shape != (len(structure), 192):
                    raise RuntimeError(f"Unexpected M3GNet feature shape for {material_id}: {node_features.shape}")
                material_cache[material_id] = structure, node_features

            structure, node_features = material_cache[material_id]
            neighbors = [
                (neighbor.index, float(neighbor.nn_distance))
                for neighbor in structure.get_neighbors(structure[site], CUTOFF)
                if float(neighbor.nn_distance) > 1e-8
            ]
            neighbor_ids = np.asarray([index for index, _ in neighbors], dtype=int)
            distance = np.asarray([value for _, value in neighbors], dtype=np.float32)
            values = node_features[neighbor_ids] if len(neighbor_ids) else np.empty((0, 192), dtype=np.float32)

            within3 = distance <= 3.0
            within5 = distance <= 5.0
            shell3_5 = (distance > 3.0) & (distance <= 5.0)
            weighted5 = np.average(values[within5], axis=0, weights=1 / distance[within5]) if within5.any() else zero_neighbors
            weighted3 = np.average(values[within3], axis=0, weights=1 / distance[within3]) if within3.any() else zero_neighbors
            weighted3_5 = np.average(values[shell3_5], axis=0, weights=1 / distance[shell3_5]) if shell3_5.any() else zero_neighbors
            rows["frozen_site_weighted5"].append(np.concatenate([node_features[site], weighted5]))
            rows["frozen_site_shells_3_5_weighted"].append(np.concatenate([node_features[site], weighted3, weighted3_5]))

            if row_number % 500 == 0:
                print(split, row_number, "/", len(split_ids[split]))

        for name, values in rows.items():
            features = np.asarray(values, dtype=np.float32)
            expected_shape = (len(split_ids[split]), FROZEN_FEATURES[name])
            if features.shape != expected_shape or not np.isfinite(features).all():
                raise RuntimeError(f"{name} {split}: expected finite {expected_shape}, got {features.shape}")
            np.savez_compressed(
                output_paths[name],
                X=features,
                ids=np.asarray(split_ids[split]),
                source=np.asarray(str(M3GNET_DIR.resolve())),
            )
            print("wrote", output_paths[name], features.shape)


In [ ]:
feature_jobs = []

for width in ENCODER_WIDTHS:
    for seed in ENCODER_SEEDS:
        run_dir = OUT_ROOT / f"learned_{width}d_seed{seed}"
        feature_file = run_dir / "features.npz"
        if not feature_file.exists():
            raise FileNotFoundError(f"Missing learned features: {feature_file}")
        with np.load(feature_file, allow_pickle=False) as saved:
            checkpoint_stamp = int(saved["checkpoint_stamp"])
            for split in SPLITS:
                if not np.array_equal(saved[f"ids_{split}"], split_ids[split]):
                    raise RuntimeError(f"ID order mismatch in {feature_file} for {split}")
            for pooling in LEARNED_POOLINGS:
                feature_jobs.append({
                    "feature": f"learned_{pooling}_{width}d",
                    "kind": "learned_encoder",
                    "member_source": f"encoder_seed{seed}",
                    "root": run_dir / pooling,
                    "feature_id": f"{run_dir.name}:{checkpoint_stamp}:{pooling}",
                    "X": {split: saved[f"{pooling}_{split}"].copy() for split in SPLITS},
                })

for name in FROZEN_FEATURES:
    paths = {split: frozen_dir / f"{name}_{split}.npz" for split in SPLITS}
    if not all(path.exists() for path in paths.values()):
        raise FileNotFoundError(f"Missing frozen features for {name}; run the frozen-feature cell")
    features = {}
    for split, path in paths.items():
        with np.load(path, allow_pickle=False) as saved:
            if not np.array_equal(saved["ids"], split_ids[split]):
                raise RuntimeError(f"ID order mismatch in {path}")
            if str(saved["source"]) != str(M3GNET_DIR.resolve()):
                raise RuntimeError(f"M3GNet provenance mismatch in {path}")
            features[split] = saved["X"].copy()
    feature_jobs.append({
        "feature": name,
        "kind": "frozen_m3gnet",
        "member_source": "frozen",
        "root": OUT_ROOT / name,
        "feature_id": f"{name}:{max(path.stat().st_mtime_ns for path in paths.values())}",
        "X": features,
    })

for job in feature_jobs:
    feature_dim = job["X"]["train"].shape[1]
    for split in SPLITS:
        if job["X"][split].shape != (len(split_ids[split]), feature_dim) or not np.isfinite(job["X"][split]).all():
            raise RuntimeError(f"Invalid feature matrix: {job['feature']} {job['member_source']} {split}")

display(pd.DataFrame([{
    "feature": job["feature"], "kind": job["kind"], "member_source": job["member_source"],
    "dim": job["X"]["train"].shape[1],
} for job in feature_jobs]))

for job in feature_jobs:
    ml_splits = MLSplits(**{split: MLData(X=job["X"][split], y=y_true[split]) for split in SPLITS})
    for head_config in HEAD_CONFIGS:
        run_name = f"seed{head_config['seed']}_dropout{head_config['dropout']:g}_lr{head_config['lr']:g}"
        head_dir = job["root"] / "heads" / run_name
        prediction_file = head_dir / "predictions.npz"
        config = {
            **head_config, "feature_id": job["feature_id"], "input_dim": job["X"]["train"].shape[1],
            "hidden_dims": HEAD_DIMS, "output_dim": 141, "epochs": HEAD_EPOCHS,
        }
        config_json = json.dumps(config, sort_keys=True)

        if prediction_file.exists():
            with np.load(prediction_file, allow_pickle=False) as saved:
                valid = str(saved["config"]) == config_json
                valid &= all(saved[split].shape == y_true[split].shape for split in ("val", "test"))
                valid &= all(np.isfinite(saved[split]).all() for split in ("val", "test"))
            if not valid:
                raise RuntimeError(f"Invalid cached predictions: {prediction_file}")
            continue
        if head_dir.exists():
            raise RuntimeError(f"Incomplete head run; remove it before retrying: {head_dir}")
        if not TRAIN_HEADS:
            raise FileNotFoundError(f"Missing head predictions: {prediction_file}")

        head_dir.mkdir(parents=True)
        pl.seed_everything(head_config["seed"], workers=True)
        XASBlock.DROPOUT = head_config["dropout"]
        regressor = XASBlockRegressor(
            directory=str(head_dir), input_dim=config["input_dim"], hidden_dims=HEAD_DIMS, output_dim=141,
            initial_lr=head_config["lr"], batch_size=BATCH_SIZE, max_epochs=HEAD_EPOCHS,
            use_early_stopping=False, use_lr_finder=False, monitor_metric="val_median_mse", shuffle=True,
            lr_scheduler="cosine", cosine_t_max=HEAD_EPOCHS, cosine_eta_min=1e-6, overwrite_save_dir=False,
        )
        regressor.fit(ml_splits).load("best")
        model = regressor.model.model.to(DEVICE).eval()
        predictions = {"config": np.asarray(config_json)}
        with torch.no_grad():
            for split in ("val", "test"):
                X = job["X"][split]
                predictions[split] = np.concatenate([
                    model(torch.as_tensor(X[start : start + 512], dtype=torch.float32, device=DEVICE)).cpu().numpy()
                    for start in range(0, len(X), 512)
                ])
                if predictions[split].shape != y_true[split].shape or not np.isfinite(predictions[split]).all():
                    raise RuntimeError(f"Invalid head predictions: {job['feature']} {run_name} {split}")
        np.savez_compressed(prediction_file, **predictions)
        print(job["feature"], job["member_source"], run_name, {
            split: round(eta(predictions[split], y_true[split]), 3) for split in ("val", "test")
        })


In [ ]:
member_rows = []
pred_val, pred_test = [], []

for job in feature_jobs:
    for head_config in HEAD_CONFIGS:
        run_name = f"seed{head_config['seed']}_dropout{head_config['dropout']:g}_lr{head_config['lr']:g}"
        prediction_file = job["root"] / "heads" / run_name / "predictions.npz"
        if not prediction_file.exists():
            raise FileNotFoundError(f"Missing configured head predictions: {prediction_file}")
        with np.load(prediction_file, allow_pickle=False) as saved:
            config = json.loads(str(saved["config"]))
            if config["feature_id"] != job["feature_id"]:
                raise RuntimeError(f"Feature provenance mismatch: {prediction_file}")
            val_prediction, test_prediction = saved["val"].copy(), saved["test"].copy()
        member_rows.append({
            "feature": job["feature"], "kind": job["kind"], "member_source": job["member_source"],
            "head": run_name, "val_eta": eta(val_prediction, y_true["val"]),
            "test_eta": eta(test_prediction, y_true["test"]),
        })
        pred_val.append(val_prediction)
        pred_test.append(test_prediction)

members = pd.DataFrame(member_rows)
pred_val, pred_test = np.stack(pred_val), np.stack(pred_test)
summary_rows = []
for feature, group in members.groupby("feature", sort=False):
    indices = group.index.to_numpy()
    best_member = group["val_eta"].idxmax()
    summary_rows.append({
        "feature": feature, "kind": group["kind"].iloc[0], "n_members": len(indices),
        "member_val_eta_mean": group["val_eta"].mean(), "member_test_eta_mean": group["test_eta"].mean(),
        "best_member_val_eta": members.loc[best_member, "val_eta"],
        "best_member_test_eta": members.loc[best_member, "test_eta"],
        "ensemble_val_eta": eta(pred_val[indices].mean(axis=0), y_true["val"]),
        "ensemble_test_eta": eta(pred_test[indices].mean(axis=0), y_true["test"]),
    })

summary = pd.DataFrame(summary_rows)
summary["delta_vs_paper_expert"] = summary["ensemble_test_eta"] - PAPER_EXPERT_ETA
summary["delta_vs_v1_ensemble"] = summary["ensemble_test_eta"] - OLD_V1_TEST_ETA
references = pd.DataFrame([
    {"feature": "paper ExpertXAS", "kind": "reference", "n_members": 1, "ensemble_val_eta": np.nan, "ensemble_test_eta": PAPER_EXPERT_ETA},
    {"feature": "old v1 20-member ensemble", "kind": "reference", "n_members": 20, "ensemble_val_eta": OLD_V1_VAL_ETA, "ensemble_test_eta": OLD_V1_TEST_ETA},
])
summary = pd.concat([references, summary], ignore_index=True, sort=False)
members.to_csv(OUT_ROOT / "members.csv", index=False)
summary.to_csv(OUT_ROOT / "summary.csv", index=False)
display(members.round(3))
display(summary.round(3))


In [ ]:
combo_masks = {
    feature: members["feature"].eq(feature).to_numpy()
    for feature in sorted(members["feature"].unique())
}
combo_masks.update({
    "learned_attention_all": members["feature"].str.startswith("learned_attention_").to_numpy(),
    "learned_weighted5_all": members["feature"].str.startswith("learned_site_weighted5_").to_numpy(),
    "learned_shells_all": members["feature"].str.startswith("learned_site_shells_3_5_weighted_").to_numpy(),
    "all_learned": members["kind"].eq("learned_encoder").to_numpy(),
    "all_frozen": members["kind"].eq("frozen_m3gnet").to_numpy(),
    "all_members": np.ones(len(members), dtype=bool),
    "best_val_member": np.arange(len(members)) == members["val_eta"].to_numpy().argmax(),
})

combo_rows = []
combo_pred_test = {}
for name, mask in combo_masks.items():
    indices = np.flatnonzero(mask)
    if not len(indices):
        continue
    val_prediction = pred_val[indices].mean(axis=0)
    test_prediction = pred_test[indices].mean(axis=0)
    combo_pred_test[name] = test_prediction
    combo_rows.append({
        "combo": name,
        "n_members": len(indices),
        "val_eta": eta(val_prediction, y_true["val"]),
        "test_eta": eta(test_prediction, y_true["test"]),
    })

# Validation eta is the only ranking criterion. Test eta is carried along for final reporting.
combo_df = pd.DataFrame(combo_rows).sort_values("val_eta", ascending=False, ignore_index=True)
combo_df.to_csv(OUT_ROOT / "combo_summary.csv", index=False)
display(combo_df.round(3))

baseline_mse = np.mean((y_true["test"] - TRAIN_MEAN) ** 2, axis=1)
best_combo = combo_df.iloc[0]["combo"]
best_mse = np.mean((y_true["test"] - combo_pred_test[best_combo]) ** 2, axis=1)
print("validation-selected combination:", best_combo)


## Visual diagnostics

The combination is fixed by validation eta before these test diagnostics are produced. The plots must not be used to revise the model or ensemble.


In [ ]:
# Validation/test agreement across individual members.
fig, ax = plt.subplots(figsize=(5.6, 4.8), dpi=150)
ax.scatter(members["val_eta"], members["test_eta"], s=45, alpha=0.8, color="#2a78d6", edgecolors="white")
low = members[["val_eta", "test_eta"]].min().min() - 0.2
high = members[["val_eta", "test_eta"]].max().max() + 0.2
ax.plot([low, high], [low, high], linestyle=":", color="gray")
ax.set(xlabel="member validation eta", ylabel="member test eta", title="Member validation/test variation")
ax.grid(alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUT_ROOT / "member_val_test_scatter.png", dpi=220)
plt.show()

# Per-spectrum errors for the validation-selected ensemble; below the diagonal is better than baseline.
if (baseline_mse <= 0).any() or (best_mse <= 0).any():
    raise ValueError("Log-error plots require positive per-spectrum MSE values")
improved = float(np.mean(best_mse < baseline_mse))
fig, ax = plt.subplots(figsize=(5.8, 5.2), dpi=150)
ax.scatter(baseline_mse, best_mse, s=14, alpha=0.5, color="#2a78d6", edgecolors="none")
low, high = min(baseline_mse.min(), best_mse.min()), max(baseline_mse.max(), best_mse.max())
ax.plot([low, high], [low, high], color="gray", linestyle=":", linewidth=1.5)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set(
    xlabel="mean-spectrum baseline MSE", ylabel=f"{best_combo} MSE",
    title=f"Validation-selected combo vs baseline ({improved:.1%} improved)",
)
ax.grid(alpha=0.25, which="both")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUT_ROOT / "validation_selected_combo_vs_baseline.png", dpi=220)
plt.show()
